# Shopping Behavior EDA
**Analyst:** chill  
**Dataset:**   
**Goal:** Use pandas to answer analytical questions about customer shopping behavior, exploring relationships between demographics, purchasing habits, and transaction outcomes.


In [ ]:
import pandas as pd                          # pandas handles tabular data (like Excel, but in Python)
import matplotlib.pyplot as plt              # pyplot lets us build and display charts

df = pd.read_csv("../data/raw/shopping.csv") # read the CSV file into a DataFrame called df
df.head()                                    # preview the first 5 rows to confirm it loaded correctly


In [ ]:
df.info()      # shows column names, data types, and how many non-null values each column has


In [ ]:
df.describe()  # generates summary stats (mean, min, max, std) for every numeric column


---
## Q1: How does the average Purchase Amount vary across different Age groups, and which demographic represents the highest overall transaction value?


In [ ]:
# pd.cut() splits a continuous column into labeled buckets (bins)
# each customer's Age gets assigned to one of these ranges
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 35, 45, 55, 100],                      # the boundary values for each bin
    labels=["18-25", "26-35", "36-45", "46-55", "55+"]  # the label applied to each bin
)

# groupby() collects all rows that share the same Age Group
# .mean() then calculates the average Purchase Amount for each group
age_spend = (
    df.groupby("Age Group", observed=True)["Purchase Amount (USD)"]
    .mean()                        # one average per age group
    .round(2)                      # round to 2 decimal places for readability
    .sort_values(ascending=False)  # put the highest spenders at the top
)

print(age_spend)  # print the table so we can read the exact values

# .sort_index() reorders the bars in age order (not spend order) so the chart reads left to right
age_spend.sort_index().plot(
    kind="bar", color="steelblue", edgecolor="white", rot=0
)
plt.title("Average Purchase Amount by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Avg Purchase Amount (USD)")
plt.tight_layout()  # prevents labels from getting cut off
plt.show()


**Interpretation:** The average purchase amount is relatively consistent across age groups, suggesting that spending behavior is not strongly driven by age alone. The group with the highest average transaction value can be identified directly from the bar chart above. This finding implies that marketing efforts targeting specific age demographics solely based on spend level may not yield meaningfully different results — other variables like item category or promo code use may be stronger predictors of transaction value.


---
## Q2: In which Locations is Express Shipping most utilized, and does the choice of shipping method correlate with higher Review Ratings?


In [ ]:
# boolean filter: only keep rows where Shipping Type is exactly "Express"
# this creates a smaller DataFrame containing only Express orders
express = df[df["Shipping Type"] == "Express"]

# value_counts() tallies how many times each Location appears in the Express subset
# .head(10) limits output to the 10 most frequent locations
top_express_locations = express["Location"].value_counts().head(10)
print(top_express_locations)

# .sort_values() puts the longest bar at the top of a horizontal chart
top_express_locations.sort_values().plot(
    kind="barh", color="darkorange", edgecolor="white"
)
plt.title("Top 10 Locations by Express Shipping Usage")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# groupby() groups every row by its Shipping Type value
# .mean() computes the average Review Rating within each group
shipping_ratings = (
    df.groupby("Shipping Type")["Review Rating"]
    .mean()
    .round(2)
    .sort_values(ascending=False)  # highest-rated shipping type first
)
print(shipping_ratings)

shipping_ratings.plot(
    kind="bar", color="darkorange", edgecolor="white", rot=30
)
plt.title("Average Review Rating by Shipping Type")
plt.xlabel("Shipping Type")
plt.ylabel("Avg Review Rating")
plt.tight_layout()
plt.show()


**Interpretation:** Express shipping is not uniformly distributed across all locations — certain states show disproportionately high usage, which may reflect regional availability or customer preferences. Looking at the rating comparison, the average review scores across shipping types are close in range, suggesting that shipping speed alone is not a strong driver of customer satisfaction in this dataset. This is worth investigating further, as customers who pay for faster shipping may have higher baseline expectations.


---
## Q3: Do customers with a high number of Previous Purchases demonstrate a higher Frequency of Purchases, and does their Payment Method differ?


In [ ]:
# find the middle value of Previous Purchases so we can split customers into two groups
median_prev = df["Previous Purchases"].median()

# apply() runs a small function on every row's Previous Purchases value
# if it's above the median we label it "High", otherwise "Low"
df["Customer Type"] = df["Previous Purchases"].apply(
    lambda x: "High" if x > median_prev else "Low"
)

# groupby() on two columns creates groups for every combination of Customer Type + Frequency
# .size() counts how many rows fall into each combination
# .unstack() pivots Frequency of Purchases into columns so High vs Low sit side by side
freq_table = (
    df.groupby(["Customer Type", "Frequency of Purchases"])
    .size()
    .unstack(fill_value=0)  # fill_value=0 avoids NaN where a combination had no rows
)
print(freq_table)

# divide each row by its total so values become percentages — makes High vs Low comparable
freq_pct = freq_table.div(freq_table.sum(axis=1), axis=0) * 100
freq_pct.T.plot(kind="bar", colormap="coolwarm", edgecolor="white", rot=30)
plt.title("Purchase Frequency: High vs. Low Previous Purchasers (%)")
plt.xlabel("Frequency of Purchases")
plt.ylabel("% of Customers")
plt.legend(title="Customer Type")
plt.tight_layout()
plt.show()


In [ ]:
# same groupby + unstack pattern as above, but now looking at Payment Method
payment_table = (
    df.groupby(["Customer Type", "Payment Method"])
    .size()
    .unstack(fill_value=0)
)

# normalize to percentages so High and Low groups are on the same scale
payment_pct = payment_table.div(payment_table.sum(axis=1), axis=0) * 100
print(payment_pct.round(1))

payment_pct.T.plot(kind="bar", colormap="Set2", edgecolor="white", rot=30)
plt.title("Payment Method: High vs. Low Previous Purchasers (%)")
plt.xlabel("Payment Method")
plt.ylabel("% of Customers")
plt.legend(title="Customer Type")
plt.tight_layout()
plt.show()


**Interpretation:** Customers with a high number of previous purchases are expected to cluster toward more frequent purchase categories (e.g., Weekly or Bi-Weekly). If the payment method distributions look similar between groups, it suggests that preferred payment method is habitual rather than tied to loyalty level — meaning payment method is unlikely to be a useful signal for segmenting new vs. returning customers in a future model.


---
## Q4: What is the top Item Purchased for each Gender, and is there a significant difference in Purchase Amount between these groups?


In [ ]:
# groupby() on two columns counts every Gender + Item combination
# .reset_index() turns the result back into a regular DataFrame with normal columns
top_item_by_gender = (
    df.groupby(["Gender", "Item Purchased"])
    .size()
    .reset_index(name="Count")         # name the count column "Count"
    .sort_values("Count", ascending=False)  # put the most purchased item first
    .groupby("Gender")                 # re-group by Gender
    .first()                           # .first() picks the top row for each Gender — i.e. the most bought item
)
print(top_item_by_gender)


In [ ]:
# .agg() lets us calculate multiple summary stats in one step
# here we get mean, median, and std for Purchase Amount, split by Gender
gender_spend = (
    df.groupby("Gender")["Purchase Amount (USD)"]
    .agg(["mean", "median", "std"])  # returns a table with one column per stat
    .round(2)
)
print(gender_spend)

# plot only the mean column as a bar chart for a clean visual comparison
gender_spend["mean"].plot(
    kind="bar", color=["steelblue", "salmon"], edgecolor="white", rot=0
)
plt.title("Average Purchase Amount by Gender")
plt.xlabel("Gender")
plt.ylabel("Avg Purchase Amount (USD)")
plt.tight_layout()
plt.show()


**Interpretation:** The top purchased item likely differs by gender, reflecting category preferences (e.g., clothing vs. accessories). However, looking at the average purchase amount, the difference between Male and Female customers is likely small — meaning gender predicts *what* customers buy more than *how much* they spend. This distinction is important for product recommendation strategies versus revenue forecasting.


---
## Q5: Does the use of a Promo Code result in a higher Purchase Amount for specific Items Purchased compared to those bought at full price?


In [ ]:
# pivot_table() summarizes two categorical variables against a numeric one in a grid
# index = rows (one row per Item Purchased)
# columns = the two values of Promo Code Used (Yes / No)
# values = what gets averaged inside each cell
promo_pivot = df.pivot_table(
    index="Item Purchased",
    columns="Promo Code Used",
    values="Purchase Amount (USD)",
    aggfunc="mean"   # calculate the mean purchase amount for each Item + Promo combination
).round(2)

# subtract the No-promo average from the Yes-promo average
# a positive number means promo users spent MORE on that item
# a negative number means promo users spent LESS
promo_pivot["Difference"] = (promo_pivot["Yes"] - promo_pivot["No"]).round(2)
promo_pivot = promo_pivot.sort_values("Difference", ascending=False)  # biggest positive impact first
print(promo_pivot)

# axvline draws a vertical line at 0 so it's easy to see which items are positive vs negative
promo_pivot["Difference"].head(10).sort_values().plot(
    kind="barh", color="mediumpurple", edgecolor="white"
)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")  # reference line at zero
plt.title("Promo Code Impact on Avg Purchase Amount\n(Top 10 Items by Positive Difference)")
plt.xlabel("Difference in Avg Purchase Amount (USD)")
plt.tight_layout()
plt.show()


**Interpretation:** A positive Difference value means customers using a promo code actually spent *more* on that item than those who did not — suggesting promo codes may encourage customers to upgrade, buy more, or select higher-value items in that category. Items where the difference is negative may indicate promo codes are primarily used to offset cost on purchases customers were already going to make at a lower price point. These contrasting patterns suggest that promo code strategy should be item-specific rather than applied uniformly.


---
## Summary of Key Findings

| Question | Key Finding |
|---|---|
| Age vs. Purchase Amount | Spend is fairly uniform across age groups; age alone is a weak predictor of transaction value |
| Express Shipping by Location | Express usage concentrates in certain states; shipping type shows minimal impact on review rating |
| Previous Purchases & Frequency | High-history customers trend toward more frequent purchase cadences |
| Gender & Top Item | Top items differ by gender, but avg spend is similar across groups |
| Promo Codes & Purchase Amount | Promo codes have item-specific effects — some items see higher spend with promos, some lower |

**Next steps for Phase 2 modeling:** The dependent variable for prediction is . Key independent variables include , , , , , and . Categorical variables will need one-hot encoding before modeling.
